# Baseline JPEG — quatre lecteurs, cinq folds communs

Ce notebook pilote les modules du dossier. **Il est livré en inspection** : « Run All »
ne lance aucun téléchargement, benchmark, extraction ou entraînement dans ce mode.
Lisez `GUIDE_FR.md` avant de sélectionner les étapes.

Les encodeurs génériques restent gelés. Les têtes apprennent douze findings avec
les labels gold et les cibles faibles, en excluant le groupe du fold évalué.
Cette adaptation aux JPEG n'est pas une reproduction des scores Kaggle publiés.

Parcours : **inspecter → préparer → vérifier → poids → benchmark → extraire train →
entraîner → évaluer → extraire test → prédire**.

## 1. Configuration et protection contre une exécution involontaire

**Prérequis :** environnement installé, notebook ouvert depuis le projet ou ce dossier.
**Sortie :** configuration en mémoire et fonction de commande ; aucun calcul d'images.

Choisissez `MODE = "smoke"` pour le parcours réduit, puis `"full"` pour les données complètes.
Ajoutez uniquement les étapes voulues à `STEPS_TO_RUN`. Le téléchargement exige aussi
`ALLOW_DOWNLOAD = True`. Les commentaires expliquent les paramètres ajustables.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Ces valeurs initiales rendent les cellules lourdes inactives, même avec Run All.
MODE = "inspection"                  # "inspection", "smoke" or "full"
STEPS_TO_RUN = set()                  # Example: {"prepare", "verify"}
ALLOW_DOWNLOAD = False                # True seulement pour votre étape "assets"
READERS_TO_RUN = "all"                 # "coatnet", "small", "tiny", "dino" ou "all"
FOLDS_TO_RUN = "all"                   # "all", "0", ou "0,1"
BENCHMARK_IMAGES = 16                  # Petit lot, jamais tout le jeu de données

candidates = [Path.cwd(), Path.cwd() / "baseline_candidate"]
PROJECT_DIR = next((p.resolve() for p in candidates if (p / "knee_baseline/config.py").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Ouvrir le notebook depuis baseline_candidate/ ou définir PROJECT_DIR explicitement.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from knee_baseline.config import load_config, output_root

if MODE not in {"inspection", "smoke", "full"}:
    raise ValueError("MODE doit être inspection, smoke ou full.")
CONFIG_PATH = PROJECT_DIR / "configs" / ("smoke.json" if MODE == "smoke" else "full.json")
CFG = load_config(CONFIG_PATH)
print("Mode :", MODE, "| profil :", CFG["profile"], "| sorties :", output_root(CFG))


def run_step(step, *arguments):
    """Exécute seulement une étape sélectionnée ; inspect reste une lecture légère."""
    if step != "inspect" and (MODE == "inspection" or step not in STEPS_TO_RUN):
        print(f"Étape {step} inactive : sélectionner le mode et STEPS_TO_RUN pour la lancer.")
        return
    if step == "assets" and not ALLOW_DOWNLOAD:
        print("Téléchargement inactif : mettre ALLOW_DOWNLOAD=True pour lancer assets.")
        return
    command = [sys.executable, "-m", "knee_baseline", "--config", str(CONFIG_PATH), *arguments]
    print("Commande :", " ".join(command))
    # Pas de shell : les chemins contenant des espaces restent des arguments corrects.
    result = subprocess.run(command, cwd=PROJECT_DIR)
    if result.returncode:
        raise RuntimeError(f"Échec de l'étape {step}, code {result.returncode}. Voir le message précédent.")

## 2. Inspecter les CSV

**Prérequis :** Aucun poids ou JPEG nécessaire.

**Sortie :** Effectifs train/gold/weak/test ; aucune écriture de résultats.

In [ ]:
# Cette cellule ne démarre que si son étape est explicitement activée.
run_step("inspect", "inspect")

## 3. Préparer les labels, folds et images

**Prérequis :** CSV et JPEG disponibles dans le data_dir configuré.

**Sortie :** prepared/ : master des folds, cibles, images sélectionnées et diagnostics.

In [ ]:
# Cette cellule ne démarre que si son étape est explicitement activée.
run_step("prepare", "prepare")

## 4. Vérifier les entrées et les caches existants

**Prérequis :** Préparation terminée. Les poids et caches peuvent être encore absents.

**Sortie :** Contrôles des empreintes ; les traitements manquants ne sont pas lancés.

In [ ]:
# Cette cellule ne démarre que si son étape est explicitement activée.
run_step("verify", "verify")

## 5. Télécharger explicitement les poids génériques

**Prérequis :** Internet et ALLOW_DOWNLOAD=True. Téléchargement potentiellement volumineux.

**Sortie :** assets/ : fichiers des poids, révisions exactes et SHA-256.

In [ ]:
# Cette cellule ne démarre que si son étape est explicitement activée.
run_step("assets", "assets", "--reader", READERS_TO_RUN)

## 6. Mesurer le temps sur quelques images

**Prérequis :** Préparation et poids du lecteur disponibles.

**Sortie :** reports/benchmark.json : vitesse et estimation indicative de l’extraction train.

In [ ]:
# Cette cellule ne démarre que si son étape est explicitement activée.
run_step("benchmark", "benchmark", "--reader", READERS_TO_RUN, "--images", str(BENCHMARK_IMAGES))

## 7. Extraire les caractéristiques train

**Prérequis :** Préparation et poids disponibles. Peut durer longtemps sur CPU.

**Sortie :** features/<reader>/train/ : [6,K,D] float16, masques [6,K] et manifeste reprenable.

In [ ]:
# Cette cellule ne démarre que si son étape est explicitement activée.
run_step("extract_train", "extract", "--reader", READERS_TO_RUN, "--split", "train")

## 8. Entraîner les têtes sur les folds choisis

**Prérequis :** Cache train complet. Les encodeurs gelés ne sont plus exécutés.

**Sortie :** checkpoints/, pertes train et probabilités OOF des folds terminés.

In [ ]:
# Cette cellule ne démarre que si son étape est explicitement activée.
run_step("train", "train", "--reader", READERS_TO_RUN, "--fold", FOLDS_TO_RUN)

## 9. Comparer les lecteurs et les mélanges

**Prérequis :** Cinq folds terminés pour chacun des quatre lecteurs.

**Sortie :** reports/ : AUC gold/weak, corrélations, différences et bootstrap du candidat 5 %.

In [ ]:
# Cette cellule ne démarre que si son étape est explicitement activée.
run_step("evaluate", "evaluate")

## 10. Extraire les caractéristiques des exemples test

**Prérequis :** Préparation et poids disponibles. Le test local contient trois exemples.

**Sortie :** features/<reader>/test/ ; aucune annotation test utilisée.

In [ ]:
# Cette cellule ne démarre que si son étape est explicitement activée.
run_step("extract_test", "extract", "--reader", READERS_TO_RUN, "--split", "test")

## 11. Prédire et exporter les soumissions

**Prérequis :** Caches train/test et cinq checkpoints finaux par lecteur.

**Sortie :** Probabilités moyennes des folds, puis submissions/ avec cinq mélanges distincts.

In [ ]:
# Cette cellule ne démarre que si son étape est explicitement activée.
run_step("predict", "predict", "--reader", READERS_TO_RUN)

## 12. Lire les rapports déjà produits

**Prérequis :** un parcours evaluate terminé ; sinon cette cellule explique leur absence.
**Sortie :** aperçu des métriques existantes, sans les recalculer.

La macro AUC gold porte sur les OOF gold regroupés. Les weak mesurent seulement
l'accord avec des heuristiques. Le bootstrap compare le candidat 5 % à baseline3,
sans sélectionner de poids. Les variantes 10 % et 15 % restent exploratoires.
Les trois exemples test valident le format, jamais une performance Kaggle.

In [ ]:
import pandas as pd
from IPython.display import display

report_dir = output_root(CFG) / "reports"
auc_path = report_dir / "auc_by_finding.csv"
summary_path = report_dir / "evaluation.json"
if auc_path.exists():
    frame = pd.read_csv(auc_path)
    display(frame.loc[frame.cohort == "gold"].pivot(index="finding", columns="model", values="auc"))
    if summary_path.exists():
        display(json.loads(summary_path.read_text(encoding="utf-8")))
else:
    print("Aucun rapport créé : lancer vous-même le parcours souhaité, puis evaluate.")

## Reprise et contrôles

Pour reprendre une extraction ou un entraînement, relancer la même cellule avec
le même profil. Les caractéristiques publiées et époques terminées sont réutilisées.
Une époque interrompue recommence ; une configuration différente exige un nouveau
output_dir. Aucun checkpoint n'est choisi avec le score du fold évalué.

Les tests de livraison utilisent de petites données artificielles. Leur relevé
figure dans `VERIFICATION_FR.md`. Le notebook complet n'a pas été exécuté.
Voir `GUIDE_FR.md` pour les commandes terminal, les fichiers produits et le dépannage.